# Customer Journey POC — run cells top to bottom
Synthetic learning data only. Requires serverless Python notebook and a writable Unity Catalog catalog.
This notebook overwrites only this project's `cj_*` demo tables. One fixed UTC snapshot;
not a production incremental pipeline. Do not run concurrently with the demo assistant.
Run `02_metric_view.ipynb` after this notebook, then follow `docs/BUILD_GUIDE.md` for Genie/MCP.


## 1. Choose catalog and create schemas

In [ ]:
# Set this to an existing writable catalog from Catalog Explorer.
CATALOG = "workspace"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
spark.sql(f"USE CATALOG `{CATALOG}`")
spark.conf.set("spark.sql.session.timeZone", "UTC")
for schema in ["cj_bronze", "cj_silver", "cj_gold", "cj_graph", "cj_semantic"]:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{schema}`")
print("Catalog:", CATALOG)


## 2. Generate source records (no installs required)

In [ ]:
"""Deterministic, fictional source events. Standard library only; all timestamps UTC."""
from datetime import datetime, timedelta
import json
from pathlib import Path

AS_OF = datetime(2026, 10, 8, 0, 0, 0)
SNAPSHOT = 'demo_20261008'
SCHEMAS = {
    'customers': 'customer_id STRING, region STRING, segment STRING',
    'activation_events': 'event_id STRING, customer_id STRING, event_ts TIMESTAMP, plan_id STRING',
    'campaign_eligibility': 'event_id STRING, campaign_id STRING, eligible BOOLEAN, rule_version STRING',
    'processing_runs': 'run_id STRING, event_id STRING, processed_ts TIMESTAMP, status STRING, error_code STRING',
    'message_attempts': 'message_id STRING, event_id STRING, campaign_id STRING, sent_ts TIMESTAMP, provider_status STRING',
    'delivery_receipts': 'receipt_id STRING, message_id STRING, delivered_ts TIMESTAMP, delivery_status STRING',
}
KEYS = dict(zip(SCHEMAS, ['customer_id', 'event_id', None, 'run_id', 'message_id', 'receipt_id']))

def generate():
    tables = {name: [] for name in SCHEMAS}
    truth = []
    for i in range(1, 10041):
        customer, event = f'C{i:05}', f'E{i:05}'
        ts = datetime(2026, 10, 1) + timedelta(seconds=(i * 53) % (7 * 86400))
        outcome, cause = 'ON_TIME', 'NOT_APPLICABLE'
        if 9501 <= i <= 9700:
            outcome, cause = 'LATE', 'UNKNOWN'
        elif 9701 <= i <= 9880:
            outcome, cause = 'NO_CONFIRMED_DELIVERY', 'VALIDATION_FAILURE'
        elif 9881 <= i <= 9970:
            outcome, cause = 'NO_CONFIRMED_DELIVERY', 'PROVIDER_REJECTION'
        elif 9971 <= i <= 10000:
            outcome, cause = 'NO_CONFIRMED_DELIVERY', 'UNKNOWN'
        elif 10001 <= i <= 10020:
            ts = AS_OF - timedelta(minutes=5)
            outcome, cause = 'PENDING', 'NOT_APPLICABLE'
        elif i > 10020:
            outcome, cause = 'INELIGIBLE', 'NOT_APPLICABLE'
        tables['customers'].append(dict(customer_id=customer, region=['North','South','East','West'][(i-1)%4], segment='PREPAID'))
        tables['activation_events'].append(dict(event_id=event, customer_id=customer, event_ts=ts, plan_id='PLAN_499'))
        tables['campaign_eligibility'].append(dict(event_id=event, campaign_id='WELCOME', eligible=outcome!='INELIGIBLE', rule_version='v1'))
        truth.append(dict(opportunity_id=f'{event}:WELCOME', customer_id=customer, outcome=outcome, expected_cause=cause))
        if outcome in ('PENDING','INELIGIBLE'):
            continue
        # First 100 recover from a failure; must not be counted as missed.
        if i <= 100:
            tables['processing_runs'].append(dict(run_id=f'R{i:05}_0', event_id=event, processed_ts=ts+timedelta(minutes=1), status='FAILED', error_code='TEMPORARY_ERROR'))
        failed = cause == 'VALIDATION_FAILURE'
        tables['processing_runs'].append(dict(run_id=f'R{i:05}_1', event_id=event, processed_ts=ts+timedelta(minutes=2), status='FAILED' if failed else 'SUCCESS', error_code='MISSING_PHONE' if failed else None))
        if failed:
            continue
        message = f'M{i:05}_1'
        tables['message_attempts'].append(dict(message_id=message, event_id=event, campaign_id='WELCOME', sent_ts=ts+timedelta(minutes=3), provider_status='REJECTED' if cause=='PROVIDER_REJECTION' else 'ACCEPTED'))
        if outcome in ('ON_TIME','LATE'):
            delay = 30 if outcome=='LATE' else (15 if i==9500 else 5)
            receipt = dict(receipt_id=f'D{i:05}_1', message_id=message, delivered_ts=ts+timedelta(minutes=delay), delivery_status='DELIVERED')
            tables['delivery_receipts'].append(receipt)
            if i<=25:
                # Distinct callback ID for the same delivery; earliest delivery wins.
                tables['delivery_receipts'].append({**receipt, 'receipt_id': f'D{i:05}_2'})
    # Exact source duplicates exercise Silver de-duplication.
    for name in tables:
        tables[name].append(dict(tables[name][0]))
    return tables, truth



In [ ]:
tables, truth = generate()
for name, rows in tables.items():
    df = spark.createDataFrame(rows, SCHEMAS[name])
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"cj_bronze.{name}")
    # Exact duplicates only. Real conflicting revisions require event/version ordering.
    silver = df.dropDuplicates()
    key = ["event_id", "campaign_id"] if name == "campaign_eligibility" else [KEYS[name]]
    assert silver.groupBy(*key).count().filter("count > 1").count() == 0, f"Conflicting keys in {name}"
    silver.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"cj_silver.{name}")
    print(name, len(rows), "raw rows")
# Truth is for testing only, never exposed to Genie or investigation logic.
spark.createDataFrame(truth).write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("cj_bronze.expected_outcomes")


## 3. Build one row per expected communication

In [ ]:
spark.sql("CREATE OR REPLACE TABLE cj_gold.communication_outcomes USING DELTA AS\nWITH first_delivery AS (\n  SELECT m.event_id, m.campaign_id, MIN(r.delivered_ts) AS first_delivered_ts\n  FROM cj_silver.message_attempts m\n  JOIN cj_silver.delivery_receipts r ON m.message_id = r.message_id\n  WHERE r.delivery_status = 'DELIVERED'\n    AND r.delivered_ts <= TIMESTAMP '2026-10-08 00:00:00'\n    AND m.sent_ts <= TIMESTAMP '2026-10-08 00:00:00'\n  GROUP BY m.event_id, m.campaign_id\n)\nSELECT concat(e.event_id, ':', q.campaign_id) AS opportunity_id,\n e.customer_id, e.event_id, q.campaign_id, c.region, c.segment,\n e.event_ts AS activation_ts, e.event_ts + INTERVAL 15 MINUTES AS deadline_ts,\n d.first_delivered_ts,\n CASE\n   WHEN d.first_delivered_ts <= e.event_ts + INTERVAL 15 MINUTES THEN 'ON_TIME'\n   WHEN d.first_delivered_ts IS NOT NULL THEN 'LATE'\n   WHEN e.event_ts + INTERVAL 15 MINUTES > TIMESTAMP '2026-10-08 00:00:00' THEN 'PENDING'\n   ELSE 'NO_CONFIRMED_DELIVERY'\n END AS outcome,\n 'demo_20261008' AS snapshot_id,\n TIMESTAMP '2026-10-08 00:00:00' AS analysis_as_of_ts\nFROM cj_silver.activation_events e\nJOIN cj_silver.customers c ON e.customer_id = c.customer_id\nJOIN cj_silver.campaign_eligibility q ON e.event_id = q.event_id AND q.eligible = true\nLEFT JOIN first_delivery d ON e.event_id = d.event_id AND q.campaign_id = d.campaign_id\nWHERE e.event_ts <= TIMESTAMP '2026-10-08 00:00:00'")

## 4. Build graph and evidence view

In [ ]:
spark.sql("CREATE OR REPLACE TABLE cj_graph.edges USING DELTA AS\nSELECT concat('customer:', customer_id) src_id, 'TRIGGERED' relationship,\n concat('event:', event_id) dst_id, 'activation_events' evidence_table, event_id evidence_id,\n event_ts observed_at, 'demo_20261008' snapshot_id\nFROM cj_silver.activation_events WHERE event_ts <= TIMESTAMP '2026-10-08 00:00:00'\nUNION ALL\nSELECT concat('event:',event_id), 'PROCESSED_BY', concat('run:',run_id),\n 'processing_runs',run_id,processed_ts,'demo_20261008'\nFROM cj_silver.processing_runs WHERE processed_ts <= TIMESTAMP '2026-10-08 00:00:00'\nUNION ALL\nSELECT concat('run:',run_id), 'FAILED_WITH', concat('error:',error_code),\n 'processing_runs',run_id,processed_ts,'demo_20261008'\nFROM cj_silver.processing_runs WHERE status='FAILED' AND processed_ts <= TIMESTAMP '2026-10-08 00:00:00'\nUNION ALL\nSELECT concat('event:',event_id), 'GENERATED', concat('message:',message_id),\n 'message_attempts',message_id,sent_ts,'demo_20261008'\nFROM cj_silver.message_attempts WHERE sent_ts <= TIMESTAMP '2026-10-08 00:00:00'\nUNION ALL\nSELECT concat('message:',message_id), 'HAS_RECEIPT', concat('receipt:',receipt_id),\n 'delivery_receipts',receipt_id,delivered_ts,'demo_20261008'\nFROM cj_silver.delivery_receipts WHERE delivered_ts <= TIMESTAMP '2026-10-08 00:00:00'")

In [ ]:
spark.sql("CREATE OR REPLACE TABLE cj_graph.nodes USING DELTA AS\nSELECT node_id, split(node_id, ':')[0] node_type, snapshot_id\nFROM (\n SELECT src_id node_id, snapshot_id FROM cj_graph.edges\n UNION\n SELECT dst_id node_id, snapshot_id FROM cj_graph.edges\n)")

In [ ]:
spark.sql("CREATE OR REPLACE VIEW cj_graph.failure_paths AS\nSELECT a.src_id event_node, a.dst_id run_node, b.dst_id error_node,\n b.evidence_id, b.observed_at, a.snapshot_id\nFROM cj_graph.edges a JOIN cj_graph.edges b\n ON a.dst_id=b.src_id AND a.snapshot_id=b.snapshot_id\nWHERE a.relationship='PROCESSED_BY' AND b.relationship='FAILED_WITH'")

In [ ]:
spark.sql("CREATE OR REPLACE VIEW cj_gold.investigation AS\nWITH latest_run AS (\n SELECT *, row_number() OVER (PARTITION BY event_id ORDER BY processed_ts DESC, run_id DESC) rn\n FROM cj_silver.processing_runs WHERE processed_ts <= TIMESTAMP '2026-10-08 00:00:00'\n), latest_message AS (\n SELECT *, row_number() OVER (PARTITION BY event_id,campaign_id ORDER BY sent_ts DESC,message_id DESC) rn\n FROM cj_silver.message_attempts WHERE sent_ts <= TIMESTAMP '2026-10-08 00:00:00'\n)\nSELECT o.*, r.run_id latest_run_id, r.status latest_run_status, r.error_code,\n m.message_id latest_message_id, m.provider_status,\n p.error_node, p.evidence_id graph_evidence_id,\n CASE\n WHEN o.outcome IN ('ON_TIME','PENDING') THEN 'NOT_APPLICABLE'\n WHEN o.outcome='NO_CONFIRMED_DELIVERY' AND r.status='FAILED'\n      AND p.error_node='error:MISSING_PHONE' AND m.message_id IS NULL THEN 'VALIDATION_FAILURE'\n WHEN o.outcome='NO_CONFIRMED_DELIVERY' AND m.provider_status='REJECTED' THEN 'PROVIDER_REJECTION'\n ELSE 'UNKNOWN' END finding\nFROM cj_gold.communication_outcomes o\nLEFT JOIN latest_run r ON o.event_id=r.event_id AND r.rn=1\nLEFT JOIN latest_message m ON o.event_id=m.event_id AND o.campaign_id=m.campaign_id AND m.rn=1\nLEFT JOIN cj_graph.failure_paths p ON p.event_node=concat('event:',o.event_id)\n AND p.evidence_id=r.run_id AND p.snapshot_id=o.snapshot_id")

## 5. Create fallback KPI view

In [ ]:
spark.sql("CREATE OR REPLACE VIEW cj_semantic.communication_kpis AS\nSELECT campaign_id, snapshot_id,\n count(*) expected_communications,\n sum(CASE WHEN outcome='ON_TIME' THEN 1 ELSE 0 END) on_time_communications,\n sum(CASE WHEN outcome='LATE' THEN 1 ELSE 0 END) late_communications,\n sum(CASE WHEN outcome='NO_CONFIRMED_DELIVERY' THEN 1 ELSE 0 END) unconfirmed_communications,\n sum(CASE WHEN outcome IN ('LATE','NO_CONFIRMED_DELIVERY') THEN 1 ELSE 0 END) sla_breaches,\n sum(CASE WHEN outcome IN ('LATE','NO_CONFIRMED_DELIVERY') THEN 1.0 ELSE 0.0 END)/NULLIF(count(*),0) sla_breach_rate\nFROM cj_gold.communication_outcomes WHERE outcome<>'PENDING'\nGROUP BY campaign_id,snapshot_id")

## 6. Verify outcomes, retries, evidence and graph integrity

In [ ]:
from collections import Counter
outcomes = spark.table("cj_gold.communication_outcomes")
actual = {r.opportunity_id:r.outcome for r in outcomes.select("opportunity_id","outcome").collect()}
expected = {r["opportunity_id"]:r["outcome"] for r in truth if r["outcome"] != "INELIGIBLE"}
assert actual == expected
assert outcomes.count() == len(actual) == 10020
assert dict(Counter(actual.values())) == {"ON_TIME":9500,"LATE":200,"NO_CONFIRMED_DELIVERY":300,"PENDING":20}
assert actual["E00001:WELCOME"] == "ON_TIME"  # Recovered failure
assert actual["E09500:WELCOME"] == "ON_TIME"  # Exactly at SLA boundary
investigation = spark.table("cj_gold.investigation")
assert investigation.count() == 10020
findings = {r.opportunity_id:r.finding for r in investigation.select("opportunity_id","finding").collect()}
assert findings == {r["opportunity_id"]:r["expected_cause"] for r in truth if r["outcome"] != "INELIGIBLE"}
assert spark.sql("SELECT COUNT(*) n FROM cj_graph.failure_paths").first().n == 280
assert spark.sql("SELECT COUNT(*) n FROM cj_graph.edges e LEFT ANTI JOIN cj_graph.nodes n ON e.dst_id=n.node_id AND e.snapshot_id=n.snapshot_id").first().n == 0
kpi = spark.table("cj_semantic.communication_kpis").first()
assert kpi.expected_communications == 10000 and kpi.sla_breaches == 500
assert abs(float(kpi.sla_breach_rate)-0.05) < 1e-9
print("PASS: all outcome, evidence, retry, duplicate, graph and KPI checks")
display(spark.table("cj_semantic.communication_kpis"))


## 7. Inspect example journey and findings

In [ ]:
display(spark.sql("SELECT * FROM cj_gold.investigation WHERE customer_id IN ('C00001','C09701','C09881','C09971','C10001') ORDER BY customer_id"))
display(spark.sql("SELECT * FROM cj_graph.failure_paths WHERE event_node='event:E09701'"))
